In [2]:
import re
import ftfy
import torch
import unicodedata

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [4]:
MODEL_PATH = "../model/indobert_best"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.to(device)

model.eval()

print("Model berhasil dimuat.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model berhasil dimuat.


In [5]:
normalisasi = {

    "gk":"tidak",
    "ga":"tidak",
    "gak":"tidak",
    "nggak":"tidak",
    "enggak":"tidak",

    "yg":"yang",
    "aja":"saja",
    "udh":"sudah",
    "sdh":"sudah",
    "blm":"belum",
    "bgt":"banget",
    "trs":"terus",
    "tp":"tapi",
    "utk":"untuk",
    "jd":"jadi",
    "klu":"kalau",
    "klo":"kalau",
    "dr":"dari",
    "dgn":"dengan",

    "gw":"saya",
    "gue":"saya",
    "sy":"saya",
    "lu":"kamu",
    "loe":"kamu",

    "org":"orang",
    "krn":"karena",
    "sm":"sama",
    "min":"admin"
}

mapping = {

    "А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H",
    "О":"O","Р":"P","С":"C","Т":"T","Х":"X","У":"Y",

    "а":"a","е":"e","о":"o","р":"p","с":"c",
    "х":"x","у":"y","к":"k","м":"m","н":"h",

    "Α":"A","Β":"B","Ε":"E","Η":"H",
    "Ι":"I","Κ":"K","Μ":"M","Ν":"N",
    "Ο":"O","Ρ":"P","Τ":"T","Χ":"X"

}

TRANSLATE_TABLE = str.maketrans(mapping)

HTML_RE = re.compile(r"<.*?>")
URL_RE = re.compile(r"http\S+")
WWW_RE = re.compile(r"www\S+")
MENTION_RE = re.compile(r"@\w+")
EMOJI_RE = re.compile(r"[\U00010000-\U0010ffff]", flags=re.UNICODE)
SPECIAL_RE = re.compile(r"[^A-Za-z0-9\s]")
SPACE_RE = re.compile(r"\s+")


def preprocess(text):

    text = str(text)

    text = ftfy.fix_text(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(TRANSLATE_TABLE)

    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = WWW_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = SPECIAL_RE.sub(" ", text)

    text = text.lower()

    words = [
        normalisasi.get(word, word)
        for word in text.split()
    ]

    text = " ".join(words)

    text = SPACE_RE.sub(" ", text)

    return text.strip()

In [6]:
import torch.nn.functional as F

MAX_LENGTH = 128

label_mapping = {
    0: "Bukan Judi Online",
    1: "Judi Online"
}

def predict(text):

    clean_text = preprocess(text)

    encoding = tokenizer(

        clean_text,

        return_tensors="pt",

        padding="max_length",

        truncation=True,

        max_length=MAX_LENGTH

    )

    input_ids = encoding["input_ids"].to(device)

    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        probabilities = F.softmax(outputs.logits, dim=1)

        prediction = torch.argmax(probabilities).item()

    print("="*60)

    print("Input")

    print(text)

    print()

    print("Preprocessing")

    print(clean_text)

    print()

    print("Prediksi")

    print(label_mapping[prediction])

    print()

    print("Confidence")

    print(f"{probabilities[0][prediction].item()*100:.2f}%")

    print("="*60)

In [7]:
predict(
    "slot gacor hari ini deposit 10 ribu langsung maxwin"
)

Input
slot gacor hari ini deposit 10 ribu langsung maxwin

Preprocessing
slot gacor hari ini deposit 10 ribu langsung maxwin

Prediksi
Judi Online

Confidence
98.54%


In [8]:
predict(
    "Selamat pagi semuanya semoga sehat selalu"
)

Input
Selamat pagi semuanya semoga sehat selalu

Preprocessing
selamat pagi semuanya semoga sehat selalu

Prediksi
Bukan Judi Online

Confidence
92.54%


In [9]:
predict(
    "Admin tolong cek pesanan saya belum dikirim"
)

Input
Admin tolong cek pesanan saya belum dikirim

Preprocessing
admin tolong cek pesanan saya belum dikirim

Prediksi
Bukan Judi Online

Confidence
98.47%
